[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-REO/blob/main/N2N-REO.ipynb)

In [ ]:
gpu_id = 1

In [ ]:
# %pip install numpy
import numpy as np

In [ ]:
sliice = np.s_[600:1200, 600:1200]

# N2N-REO (Noise2Noise Registered Even Odd denoising)

## Packages

In [ ]:
# pip install mrcfile # (install NumPy)
import mrcfile

In [ ]:
# pip install numpy
import numpy as np

In [ ]:
# pip install tqdm ipywidgets
from tqdm.notebook import tqdm

In [ ]:
# pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
# pip install opencv-python
import cv2

In [ ]:
import json

In [ ]:
# pip install tensorflow


In [ ]:
# pip install cryoCARE --no-deps

In [ ]:
# pip install csbdeep

In [ ]:
# pip install gdown
import gdown

## Download a noisy tomogram

In [ ]:
url="https://drive.google.com/file/d/1VSWgd8xS4zlw6amzTdqz5OdsnGh6aoCo"
#gdown.download(url, output="000.mrc", quiet=False, use_cookies=False)

In [ ]:
X = mrcfile.open("000.mrc").data

In [ ]:
X.shape

## Split the tomogram in even and odd axial slices

In [ ]:
even_vol = X[0::2,:,:]
with mrcfile.new("even.mrc", overwrite=True) as mrc:
    mrc.set_data(even_vol)
    mrc.data

In [ ]:
even_vol.shape

In [ ]:
!ls -l "even.mrc"

In [ ]:
odd_vol = X[1::2,:,:]
with mrcfile.new("odd.mrc", overwrite=True) as mrc:
    mrc.set_data(odd_vol)
    mrc.data
    mrc.data

In [ ]:
odd_vol.shape

In [ ]:
!ls -l "odd.mrc"

## Register the even and odd tomograms by axial slices

In [ ]:
farneback_params = dict(
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0
)

In [ ]:
projected_vol = np.zeros_like(odd_vol, dtype=np.float32)

In [ ]:
for z in tqdm(range(even_vol.shape[0]), desc="Projecting Slices"):

    # Calculate the dense optical flow from slice_z_plus_1 to slice_z
    flow = cv2.calcOpticalFlowFarneback(even_vol[z, ...], odd_vol[z, ...], None, **farneback_params)

    # Create a remapping grid from the flow field
    height, width = flow.shape[:2]
    x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))

    # The new map tells where each pixel in the output image should come from in the input image
    map_x = (x_coords + flow[..., 0]).astype(np.float32)
    map_y = (y_coords + flow[..., 1]).astype(np.float32)

    # Warp the *original float32 slice* using the map for maximum precision
    original_slice_to_warp = odd_vol[z , ...]
    projected_slice = cv2.remap(
        src=original_slice_to_warp,
        map1=map_x,
        map2=map_y,
        #interpolation=cv2.INTER_LINEAR,
        interpolation=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_REPLICATE # Handle edge pixels
    )

    # Store the result
    projected_vol[z, ...] = projected_slice

In [ ]:
projected_vol.shape

In [ ]:
slice_idx = even_vol.shape[0] // 2

fig, axes = plt.subplots(1, 5, figsize=(20, 20))

im1 = axes[0].imshow(even_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Even z={slice_idx}')
axes[0].grid(False)

im2 = axes[1].imshow(odd_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[1].set_title(f'Original Slice Odd z={slice_idx}')
axes[1].grid(False)

im3 = axes[2].imshow(projected_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[2].set_title(f'Projected Slice (Odd[z] -> Even[z])')
axes[2].grid(False)

im4 = axes[3].imshow((even_vol[slice_idx][sliice].T - odd_vol[slice_idx][sliice].T + 128).astype(np.int16), cmap='gray', origin='lower')
axes[3].set_title(f'even[z] - odd[z]')
axes[3].grid(False)

im5 = axes[4].imshow((even_vol[slice_idx][sliice].T - projected_vol[slice_idx][sliice].T + 128).astype(np.int16), cmap='gray', origin='lower')
axes[4].set_title(f'even[z] - odd_projected[z]')
axes[4].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
output_filename = 'odd_registered.mrc'
with mrcfile.new(output_filename, overwrite=True) as mrc:
    mrc.set_data(projected_vol)
    mrc.data
    mrc.data

In [ ]:
!ls -l odd_registered.mrc

In [ ]:
for z in tqdm(range(even_vol.shape[0]), desc="Projecting Slices"):

    # Calculate the dense optical flow from slice_z_plus_1 to slice_z
    flow = cv2.calcOpticalFlowFarneback(odd_vol[z, ...], even_vol[z, ...], None, **farneback_params)

    # Create a remapping grid from the flow field
    height, width = flow.shape[:2]
    x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))

    # The new map tells where each pixel in the output image should come from in the input image
    map_x = (x_coords + flow[..., 0]).astype(np.float32)
    map_y = (y_coords + flow[..., 1]).astype(np.float32)

    # Warp the *original float32 slice* using the map for maximum precision
    original_slice_to_warp = even_vol[z , ...]
    projected_slice = cv2.remap(
        src=original_slice_to_warp,
        map1=map_x,
        map2=map_y,
        #interpolation=cv2.INTER_LINEAR,
        interpolation=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_REPLICATE # Handle edge pixels
    )

    # Store the result
    projected_vol[z, ...] = projected_slice

In [ ]:
projected_vol.shape

In [ ]:
slice_idx = even_vol.shape[0] // 2

fig, axes = plt.subplots(1, 5, figsize=(20, 20))

im1 = axes[0].imshow(odd_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Odd z={slice_idx}')
axes[0].grid(False)

im2 = axes[1].imshow(even_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[1].set_title(f'Original Slice Even z={slice_idx}')
axes[1].grid(False)

im3 = axes[2].imshow(projected_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[2].set_title(f'Projected Slice (Even[z] -> Odd[z])')
axes[2].grid(False)

im4 = axes[3].imshow(odd_vol[slice_idx][sliice].T - even_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[3].set_title(f'odd[z] - even[z]')
axes[3].grid(False)

im5 = axes[4].imshow(odd_vol[slice_idx][sliice].T - projected_vol[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[4].set_title(f'odd[z] - even_projected[z]')
axes[4].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
output_filename = 'even_registered.mrc'
print("Writing", output_filename)

with mrcfile.new(output_filename, overwrite=True) as mrc:
    mrc.set_data(projected_vol)
    mrc.data

In [ ]:
!ls -l even_registered.mrc

# Denoising

In [ ]:
_ = {
    "even": ["even.mrc", "even_registered.mrc"],
    "odd": ["odd_registered.mrc", "odd.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_REO",
    "overwrite": "True"
}

with open("train_data_config__REO.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config__REO.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_extract_train_data.py --conf train_data_config__REO.json

In [ ]:
_ = {
  "train_data": "./data_REO",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_REO",
  "path": "./",
  "gpu_id": [gpu_id]
}
with open("train_config__REO.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_config__REO.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_train.py --conf train_config__REO.json

In [ ]:
_ = {
    "path": "./model_REO.tar.gz",
    "even": ["000.mrc"],
    "odd": ["000.mrc"],
    "n_tiles": [4,4,4],
    "output": "denoised_vol_REO",
    "overwrite": "True",
    "gpu_id": [gpu_id]
}

with open("predict_config__REO.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat predict_config__REO.json

In [ ]:
%%bash
/nas/vruiz/envs/OF3D_CUDA/bin/cryoCARE_predict.py --conf predict_config__REO.json || true

In [ ]:
!ls -l denoised_vol_REO/000.mrc

In [ ]:
Y = mrcfile.read("denoised_vol_REO/000.mrc")

import numpy as np
import mrcfile

file_path = "denoised_vol_REO/000.mrc"

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
Y = data.reshape((nz, ny, nx))

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[slice_idx][sliice].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Odd-Even-Registered Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'Original Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('original.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(Y[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'N2N-REO Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('REO.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'Original Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('original_zoom.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(Y[slice_idx][sliice], cmap='gray', origin='lower')
axes.set_title(f'N2N-REO Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('REO_zoom.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[1] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, slice_idx, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Y={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[:, slice_idx, :], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-REO Slice Y={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, :, slice_idx], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice X={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[:, :, slice_idx], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-REO Slice X={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats

In [ ]:
def PCC(original, denoised):
    return scipy.stats.pearsonr(original.flatten(), denoised.flatten())[0]

In [ ]:
print(f"PCC={PCC(X, Y):.3f}")

In [ ]:
import skimage.metrics

In [ ]:
def PSNR(original, denoised):
    return skimage.metrics.peak_signal_noise_ratio(original, denoised, data_range=original.max()-original.min())

In [ ]:
print(f"PSNR={PSNR(X, Y):.3f}")

In [ ]:
#!pip install --dry-run scikit-image
#!pip install scikit-image

def SSIM(original, denoised):
    return skimage.metrics.structural_similarity(original, denoised, data_range=original.max() - original.min())

In [ ]:
print(f"SSIM={SSIM(X, Y):.3f}")